In [ ]:
import torch
import torch.nn as nn

In [ ]:
class VGG16(nn.Module):
    global folder_df

    def __init__(self, input_shape: torch.Size, dropout_rate: float = 0):
        super().__init__()

        total_output_classes = len(folder_df["folder"].unique())

        # input shape should be some list/tuple of length 4
        if len(input_shape) != 4: return Exception("Input shape is not AxBxCxD.")

        # define layers here
        # assuming input shape = 1x1x128x126 = AxBxCxD

        # conv2d: 1x128x126 -> 1x124x124 (kernel size = (5,3))
        # relu
        # max pool: 1x124x124 -> 1x62x62 (pool size = (2,2))
        # conv2d: 1x62x62 -> 1x60x60 (kernel size = (3, 3))
        # relu
        # max pool: 1x60x60 -> 1x30x30 (pool size = (2, 2))
        # flatten: 1x30x30 -> 900
        # linear: 900 -> 128
        # linear: 128 -> 32
        # linear: 32 -> (output layers)

        A = input_shape[0]
        B = input_shape[1]
        C = input_shape[2]
        D = input_shape[3]

        # VGG-16 takes in input of size 224x224x3, but our input data is vastly different.

        self.relu = nn.ReLU()

        self.conv1 = nn.Conv2d(in_channels=B, out_channels=64, kernel_size=(3, 3), padding=2) # padding to maintain consistent shape
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), padding=2)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3, 3), padding=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3, 3), padding=2)
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3, 3), padding=2)
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3, 3), padding=2)
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        
        self.conv8 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3, 3), padding=2)
        self.conv9 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3, 3), padding=2)
        self.conv10 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3, 3), padding=2)
        self.pool4 = nn.MaxPool2d(kernel_size=2)
        
        self.conv11 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3, 3), padding=2)
        self.conv12 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3, 3), padding=2)
        self.conv13 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3, 3), padding=2)
        self.pool5 = nn.MaxPool2d(kernel_size=2)

        self.flat = nn.Flatten()
        pool_layers = 5
        flat_width = C
        flat_height = D
        while pool_layers > 0:
            flat_width = flat_width // 2
            flat_height = flat_height // 2
            pool_layers -= 1
        flat_nodes = flat_width * flat_height

        self.linear1 = nn.Linear(in_features=flat_nodes, out_features=4096)
        self.linear2 = nn.Linear(in_features=4096, out_features=4096)
        self.linear3 = nn.Linear(in_features=4096, out_features=total_output_classes)

        self.output = nn.Softmax()

    def forward(self, x):
        # define calculations here
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool1(x)
        
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.relu(self.conv5(x))
        x = self.relu(self.conv6(x))
        x = self.relu(self.conv7(x))
        x = self.pool3(x)

        x = self.relu(self.conv8(x))
        x = self.relu(self.conv9(x))
        x = self.relu(self.conv10(x))
        x = self.pool4(x)

        x = self.relu(self.conv11(x))
        x = self.relu(self.conv12(x))
        x = self.relu(self.conv13(x))
        x = self.pool5(x)

        x = self.flat(x)

        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.relu(self.linear3(x))

        x = self.output(x)

        return x